# Steam — Analyse du marché des jeux vidéo
**Contexte :** Ubisoft souhaite lancer un nouveau jeu. On analyse le marché Steam pour comprendre les tendances, les genres porteurs et ce qui fait la popularité d'un jeu.

## Setup & Chargement

In [18]:
import pandas as pd
import plotly.express as px
import urllib.request
from pathlib import Path

pd.set_option('display.max_columns', None)

In [19]:
# Téléchargement automatique du dataset si absent
local_path = Path("../data/raw/steam_game_output.json")

if not local_path.exists():
    url = "https://full-stack-bigdata-datasets.s3.eu-west-3.amazonaws.com/Big_Data/Project_Steam/steam_game_output.json"
    print("Téléchargement en cours...")
    urllib.request.urlretrieve(url, local_path)
    print("OK")
else:
    print(f"Dataset présent ({local_path.stat().st_size / 1024 / 1024:.0f} MB)")

Dataset présent (61 MB)


In [20]:
# Le JSON a une structure {id, data} — on normalise la colonne data
df_raw = pd.read_json("../data/raw/steam_game_output.json")

COLS = ['appid','name','developer','publisher','genre','type','owners',
        'positive','negative','price','initialprice','discount','ccu',
        'languages','release_date','required_age',
        'platforms.windows','platforms.mac','platforms.linux']

df = pd.json_normalize(df_raw['data'])[COLS].copy().reset_index(drop=True)

# Renommer les colonnes avec des points pour éviter les conflits
df = df.rename(columns={'platforms.windows': 'win', 'platforms.mac': 'mac', 'platforms.linux': 'linux'})

print(f"{len(df)} jeux chargés")
df.head(3)

55691 jeux chargés


,appid,name,developer,publisher,genre,type,owners,positive,negative,price,initialprice,discount,ccu,languages,release_date,required_age,win,mac,linux
0,10,Counter-Strike,Valve,Valve,Action,game,"10,000,000 .. 20,000,000",201215,5199,999,999,0,13990,"English, French, German, Italian, Spanish - Sp...",2000/11/1,0,True,True,True
1,1000000,ASCENXION,IndigoBlue Game Studio,PsychoFlux Entertainment,"Action, Adventure, Indie",game,"0 .. 20,000",27,5,999,999,0,0,"English, Korean, Simplified Chinese",2021/05/14,0,True,False,False
2,1000010,Crown Trick,NEXT Studios,"Team17, NEXT Studios","Adventure, Indie, RPG, Strategy",game,"200,000 .. 500,000",4032,646,599,1999,70,99,"Simplified Chinese, English, Japanese, Traditi...",2020/10/16,0,True,False,False


### Préparation des données

In [21]:
# Typage et calcul des colonnes utiles
df['price_eur']      = pd.to_numeric(df['price'], errors='coerce') / 100
df['discount']       = pd.to_numeric(df['discount'], errors='coerce').fillna(0).astype(int)
df['release_date']   = pd.to_datetime(df['release_date'], errors='coerce')
df['release_year']   = df['release_date'].dt.year
df['positive']       = pd.to_numeric(df['positive'], errors='coerce')
df['negative']       = pd.to_numeric(df['negative'], errors='coerce')
total_avis           = df['positive'] + df['negative']
df['positive_ratio'] = (df['positive'] / total_avis * 100).round(1)
df['required_age']   = pd.to_numeric(df['required_age'], errors='coerce').fillna(0).astype(int)

# Renommer genre_str pour libérer le nom 'genre' pour l'explode
df = df.rename(columns={'genre': 'genre_str'})

# DataFrame explosé : une ligne par couple jeu/genre
df_genres = df.copy()
df_genres['genre'] = df['genre_str'].fillna('').apply(lambda x: [g.strip() for g in x.split(',') if g.strip()])
df_genres = df_genres.explode('genre').reset_index(drop=True)
df_genres = df_genres.dropna(subset=['genre'])
df_genres = df_genres[df_genres['genre'] != ''].reset_index(drop=True)

print(f"df principal : {df.shape[0]} jeux")
print(f"df_genres    : {df_genres.shape[0]} lignes (jeux × genres)")

df principal : 55691 jeux
df_genres    : 157110 lignes (jeux × genres)


---
## 1. Analyse Macro
### Éditeurs les plus prolifiques

In [22]:
top_pub = df['publisher'].value_counts().head(20).reset_index()
top_pub.columns = ['Publisher', 'Nombre de jeux']

fig = px.bar(top_pub, x='Nombre de jeux', y='Publisher', orientation='h',
             title='Top 20 éditeurs par nombre de jeux publiés sur Steam',
             color='Nombre de jeux', color_continuous_scale='Blues', height=600)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** Big Fish Games domine largement avec plus de 400 jeux — c'est un éditeur spécialisé dans les casual games. SEGA et Square Enix sont les seuls grands studios traditionnels dans ce top 20.

### Jeux les mieux notés

In [23]:
# Filtrer sur min. 100 avis pour écarter les faux 100%
mask = total_avis >= 100
top_rated = df[mask].nlargest(15, 'positive_ratio')[
    ['name', 'publisher', 'positive_ratio', 'positive', 'release_year']
].reset_index(drop=True)

fig = px.bar(top_rated, x='positive_ratio', y='name', orientation='h',
             title="Top 15 jeux par % d'avis positifs (min. 100 avis)",
             labels={'positive_ratio': '% avis positifs', 'name': ''},
             color='positive_ratio', color_continuous_scale='Greens', height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [24]:
# Top par volume d'avis positifs brut
top_volume = df.nlargest(15, 'positive')[
    ['name', 'publisher', 'positive', 'positive_ratio', 'release_year']
].reset_index(drop=True)

fig = px.bar(top_volume, x='positive', y='name', orientation='h',
             title="Top 15 jeux par volume d'avis positifs",
             labels={'positive': "Nombre d'avis positifs", 'name': ''},
             color='positive', color_continuous_scale='Purples', height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** Les deux classements divergent : les jeux au ratio parfait sont souvent des titres de niche très appréciés. En volume, on retrouve les blockbusters comme CS:GO ou Dota 2 avec des millions d'avis.

### Sorties par année — impact du Covid ?

In [25]:
releases = df[df['release_year'].between(2000, 2024)].groupby('release_year').size().reset_index(name='nb_sorties')

fig = px.bar(releases, x='release_year', y='nb_sorties',
             title='Nombre de sorties par année sur Steam',
             labels={'release_year': 'Année', 'nb_sorties': 'Nombre de jeux'},
             color='nb_sorties', color_continuous_scale='Oranges', width=800, height=450)
fig.add_vrect(x0=2019.5, x1=2021.5, fillcolor='red', opacity=0.1,
              annotation_text='Période Covid', annotation_position='top left')
fig.show()

**Interprétation :** La croissance est spectaculaire depuis l'ouverture de Steam Direct en 2017. La période Covid (2020–2021) affiche un pic — les confinements ont boosté le marché avec +26% de sorties en 2020 vs 2019.

### Distribution des prix et promotions

In [26]:
df['type_prix'] = df['price_eur'].apply(lambda x: 'Gratuit (F2P)' if x == 0 else 'Payant')
type_prix = df['type_prix'].value_counts().reset_index()
type_prix.columns = ['Type', 'Nombre']

fig = px.pie(type_prix, values='Nombre', names='Type',
             title='Répartition jeux gratuits vs payants',
             color_discrete_map={'Gratuit (F2P)': '#2ecc71', 'Payant': '#3498db'},
             width=550, height=400)
fig.show()

In [27]:
payants = df[df['price_eur'] > 0]['price_eur']

fig = px.histogram(payants, x='price_eur', nbins=50,
                   title=f'Distribution des prix (jeux payants) — médiane : {payants.median():.2f}€',
                   labels={'price_eur': 'Prix (€)', 'count': 'Nombre de jeux'},
                   color_discrete_sequence=['#3498db'], width=750, height=430)
fig.add_vline(x=payants.median(), line_dash='dash', line_color='red',
              annotation_text=f'Médiane : {payants.median():.2f}€')
fig.show()

In [28]:
promo = df[df['price_eur'] > 0].copy()
promo['statut'] = (promo['discount'] > 0).map({True: 'En promotion', False: 'Prix normal'})
promo_counts = promo['statut'].value_counts().reset_index()
promo_counts.columns = ['Statut', 'Nombre']

fig = px.pie(promo_counts, values='Nombre', names='Statut',
             title='Part des jeux actuellement en promotion',
             color_discrete_map={'En promotion': '#e67e22', 'Prix normal': '#95a5a6'},
             width=550, height=400)
fig.show()

**Interprétation :** 86% des jeux sont payants avec une médiane à ~6€ — le marché penche vers les petits prix. Les promotions ne concernent qu'environ 5% des jeux au moment du snapshot, mais elles sont une stratégie clé sur Steam.

### Langues les plus représentées

In [29]:
lang_series = df['languages'].dropna().str.split(',').explode().str.strip()
lang_series = lang_series[lang_series != '']

top_lang = lang_series.value_counts().head(15).reset_index()
top_lang.columns = ['Langue', 'Nombre de jeux']

fig = px.bar(top_lang, x='Nombre de jeux', y='Langue', orientation='h',
             title='Top 15 langues supportées sur Steam',
             color='Nombre de jeux', color_continuous_scale='Teal', height=500)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** L'anglais est présent dans ~99% des jeux. L'allemand, le français et le russe suivent. Le chinois simplifié est 5e — un signal fort de l'importance du marché asiatique pour tout éditeur voulant maximiser sa portée.

### Restrictions d'âge

In [30]:
def tranche_age(a):
    if a == 0:    return 'Tous publics'
    elif a <= 12: return '12+'
    elif a <= 16: return '16+'
    else:         return '18+'

age_dist = df['required_age'].apply(tranche_age).value_counts().reset_index()
age_dist.columns = ['Tranche', 'Nombre']

fig = px.pie(age_dist, values='Nombre', names='Tranche',
             title="Répartition des jeux par restriction d'âge",
             width=550, height=400)
fig.show()

**Interprétation :** La quasi-totalité des jeux (99%) sont tous publics selon les métadonnées Steam. Les jeux 16+ et 18+ sont peu nombreux mais incluent souvent des titres AAA à fort budget et forte notoriété.

---
## 2. Analyse par Genre

### Genres les plus représentés

In [31]:
top_genres = df_genres['genre'].value_counts().head(20).reset_index()
top_genres.columns = ['Genre', 'Nombre de jeux']

fig = px.bar(top_genres, x='Nombre de jeux', y='Genre', orientation='h',
             title='Top 20 genres les plus représentés sur Steam',
             color='Nombre de jeux', color_continuous_scale='Blues', height=600)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** Indie domine massivement (>39k jeux) devant Action et Casual. Ces genres à faible barrière à l'entrée attirent beaucoup de petits studios. Un nouveau titre Ubisoft se positionnerait dans un segment bien plus concurrentiel.

### Ratio d'avis positifs par genre

In [32]:
mask_avis = (df_genres['positive'] + df_genres['negative']) >= 50
genre_reviews = df_genres[mask_avis].groupby('genre').agg(
    ratio_moyen=('positive_ratio', 'mean'),
    nb_jeux=('appid', 'count')
).round(1).reset_index()

genre_reviews = genre_reviews[genre_reviews['nb_jeux'] >= 20].sort_values('ratio_moyen', ascending=False)

fig = px.bar(genre_reviews, x='ratio_moyen', y='genre', orientation='h',
             title="Ratio moyen d'avis positifs par genre (min. 20 jeux, 50 avis)",
             labels={'ratio_moyen': '% avis positifs', 'genre': 'Genre'},
             color='ratio_moyen', color_continuous_scale='RdYlGn',
             range_color=[60, 90], height=600)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** Les genres créatifs (Design, Animation, Audio) ont les meilleurs ratios — leur communauté est de niche mais très satisfaite. À l'inverse, les genres Casual et Free to Play ont les ratios les plus bas, probablement à cause d'attentes plus élevées.

### Genres favoris des top éditeurs

In [33]:
top10_pub = df['publisher'].value_counts().head(10).index.tolist()

pub_genres = df_genres[df_genres['publisher'].isin(top10_pub)].groupby(
    ['publisher', 'genre']
).size().reset_index(name='nb_jeux')

fig = px.bar(pub_genres, x='nb_jeux', y='publisher', color='genre',
             title='Genres favoris des 10 plus gros éditeurs Steam',
             labels={'nb_jeux': 'Nombre de jeux', 'publisher': 'Éditeur'},
             orientation='h', height=500, width=900)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

**Interprétation :** Big Fish Games est concentré sur le Casual — cohérent avec son modèle. SEGA et Square Enix ont des catalogues plus diversifiés (RPG, Action, Adventure). Ce tableau aide Ubisoft à identifier où se positionnent ses concurrents directs.

### Prix moyen et popularité par genre

In [34]:
genre_revenue = df_genres[df_genres['price_eur'] > 0].groupby('genre').agg(
    prix_moyen=('price_eur', 'mean'),
    avis_positifs_moyens=('positive', 'mean'),
    nb_jeux=('appid', 'count')
).round(2).reset_index()

genre_revenue = genre_revenue[genre_revenue['nb_jeux'] >= 20]

fig = px.scatter(genre_revenue, x='prix_moyen', y='avis_positifs_moyens',
                 size='nb_jeux', color='genre', hover_name='genre',
                 title='Prix moyen vs popularité (avis positifs) par genre',
                 labels={'prix_moyen': 'Prix moyen (€)', 'avis_positifs_moyens': 'Avis positifs moyens'},
                 width=900, height=600)
fig.show()

**Interprétation :** Les genres avec un prix élevé ET une forte popularité (quadrant haut-droite) sont les plus lucratifs. Les genres RPG et Action se distinguent sur la popularité. Les outils créatifs (Audio, Video, Animation) ont des prix premium mais une popularité modeste.

---
## 3. Analyse par Plateforme
### Windows / Mac / Linux

In [35]:
platforms = pd.DataFrame({
    'Plateforme': ['Windows', 'Mac', 'Linux'],
    'Nombre de jeux': [df['win'].sum(), df['mac'].sum(), df['linux'].sum()],
    'Pourcentage (%)': [
        round(df['win'].mean() * 100, 1),
        round(df['mac'].mean() * 100, 1),
        round(df['linux'].mean() * 100, 1)
    ]
})

fig = px.bar(platforms, x='Plateforme', y='Pourcentage (%)',
             title='Part des jeux disponibles par plateforme',
             text='Pourcentage (%)', color='Plateforme', width=600, height=420)
fig.update_traces(texttemplate='%{text}%', textposition='outside')
fig.show()

In [36]:
def combo(row):
    p = []
    if row['win']:   p.append('Win')
    if row['mac']:   p.append('Mac')
    if row['linux']: p.append('Linux')
    return ' + '.join(p) if p else 'Aucune'

df['plateformes'] = df.apply(combo, axis=1)
combo_counts = df['plateformes'].value_counts().reset_index()
combo_counts.columns = ['Combo', 'Nombre']

fig = px.pie(combo_counts, values='Nombre', names='Combo',
             title='Combinaisons de plateformes supportées',
             width=600, height=450)
fig.show()

**Interprétation :** Windows est quasi-universel (99.97%). Mac couvre ~23% des jeux et Linux ~15%. La combinaison Win+Mac+Linux représente 12% du catalogue — les studios qui font l'effort du triple portage sont minoritaires mais visibles.

### Genres par plateforme

In [37]:
gp = df_genres.groupby('genre').agg(
    pct_win=('win', 'mean'),
    pct_mac=('mac', 'mean'),
    pct_linux=('linux', 'mean'),
    nb_jeux=('appid', 'count')
).reset_index()

gp[['pct_win', 'pct_mac', 'pct_linux']] = (gp[['pct_win', 'pct_mac', 'pct_linux']] * 100).round(1)
gp = gp[gp['nb_jeux'] >= 20].sort_values('pct_mac', ascending=False)

fig = px.bar(gp.head(20), x='genre', y=['pct_win', 'pct_mac', 'pct_linux'],
             title='Disponibilité par plateforme selon le genre (% des jeux du genre)',
             labels={'value': '% de jeux dispo', 'genre': 'Genre', 'variable': 'Plateforme'},
             barmode='group', height=500, width=1000)
fig.update_layout(xaxis_tickangle=-40)
fig.show()

**Interprétation :** Les genres créatifs (Design, Animation, Audio) et la Stratégie ont les meilleurs taux de portage Mac/Linux. Les gros genres Action et Casual sont quasi-exclusivement Windows. Un jeu de stratégie ou RPG aura une meilleure couverture multi-plateforme naturelle.

---
## Conclusion

| Axe | Enseignement clé |
|---|---|
| **Marché** | 55 000+ jeux, explosion des sorties depuis 2017, Covid = pic en 2020–2021 |
| **Prix** | Médiane à ~6€, 14% de F2P, peu de promos actives au moment du snapshot |
| **Langues** | Anglais indispensable, chinois et russe à prioriser pour l'internationalisation |
| **Genres** | Indie/Casual saturés. RPG et Action dominent en popularité. Stratégie = meilleur ratio qualité/prix |
| **Plateformes** | Windows incontournable. Mac/Linux = avantage différenciant pour Stratégie/RPG |

**Recommandation Ubisoft :** se positionner sur un RPG ou jeu de Stratégie avec pricing ≥ 15€, support anglais + chinois + russe dès le lancement, et portage Windows + Mac pour toucher un public plus large.